# Multimodal Fake News Detection — Colab GPU training
Run cells in order. Select **Runtime → Change runtime type → T4 GPU** first.
Only pretrained encoder weights and Python dependencies are downloaded; **Fakeddit is never downloaded**.
Use your existing 3,000/500/500 processed pairs. Backbones remain frozen; only classification heads train.
Drive stores cached features, epoch recovery, models, and results. Keep the same settings to resume after disconnecting.
Upload this notebook to Colab; it embeds all required Python source, so no repository upload is needed.


## 1. Install dependencies (keep Colab's matched CUDA torch/torchvision)

In [ ]:
%pip -q install "transformers==4.57.1" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "Pillow>=10,<13" safetensors
import torch, torchvision
assert torch.cuda.is_available(), "Enable a Colab GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__, "torchvision:", torchvision.__version__)


## 2. Mount Drive and write the shared training/inference code

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, json, shutil, sys
drive.mount('/content/drive')
WORK = Path('/content/fakeddit_code')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
SOURCE_FILES = {'dataset.py': '"""Read existing paired data without rewriting CSVs or downloading images."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport pandas as pd\nfrom PIL import Image\n\nSIZES = {"train": 3000, "validate": 500, "test": 500}\nLABELS = {0: "Fake", 1: "Real"}\n\n\ndef resolve_image(root, raw):\n    root = Path(root).resolve()\n    parts = str(raw).replace("\\\\", "/").split("/")\n    # Relocate even absolute Windows paths by the images/ suffix.\n    if "images" not in parts:\n        raise ValueError(f"Image path has no images directory: {raw}")\n    relative = Path(*parts[parts.index("images"):])\n    target = (root / relative).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Unsafe image path: {raw}")\n    return target\n\n\ndef extract_zip(archive, destination):\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for entry in z.infolist():\n            name = entry.filename.replace("\\\\", "/")\n            target = (destination / name).resolve()\n            if not target.is_relative_to(destination) or ":" in name:\n                raise ValueError(f"Unsafe ZIP member: {name}")\n            if (entry.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are not supported")\n        for entry in z.infolist():\n            target = destination / entry.filename.replace("\\\\", "/")\n            if entry.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(entry) as src, target.open("wb") as dst:\n                    import shutil\n                    shutil.copyfileobj(src, dst)\n    matches = list(destination.rglob("train_processed.csv"))\n    if len(matches) != 1:\n        raise ValueError("ZIP must contain exactly one processed dataset")\n    return matches[0].parent\n\n\ndef validate(root, report_path):\n    frames, report, errors = {}, {"label_mapping": LABELS, "splits": {}}, []\n    for split, expected in SIZES.items():\n        df = pd.read_csv(Path(root) / f"{split}_processed.csv", dtype={"id": str}, keep_default_na=False)\n        required = ["id", "clean_title", "2_way_label", "local_image_path"]\n        if not set(required).issubset(df.columns):\n            raise ValueError(f"Missing columns in {split}: {set(required) - set(df.columns)}")\n        missing = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in required}\n        duplicates = df.loc[df.id.duplicated(), "id"].tolist()\n        if len(df) != expected or any(missing.values()) or duplicates:\n            errors.append(f"{split}: count/missing/duplicate failure")\n        numeric = pd.to_numeric(df["2_way_label"], errors="coerce")\n        if not numeric.isin([0, 1]).all():\n            errors.append(f"{split}: labels must be 0 or 1")\n        if "6_way_label" in df:\n            six = pd.to_numeric(df["6_way_label"], errors="coerce")\n            if not six.isin(range(6)).all() or not ((six == 0) == (numeric == 1)).all():\n                errors.append(f"{split}: binary labels inconsistent with 6-way true category")\n        paths, hashes, bad = [], [], []\n        for row in df.itertuples(index=False):\n            try:\n                p = resolve_image(root, row.local_image_path)\n                with Image.open(p) as im:\n                    im.verify()\n                with Image.open(p) as im:\n                    im.convert("RGB").load()\n                hashes.append(hashlib.sha256(p.read_bytes()).hexdigest())\n                paths.append(str(p))\n            except Exception as exc:\n                bad.append({"id": row.id, "error": str(exc)})\n                paths.append("")\n                hashes.append("")\n        if bad:\n            errors.append(f"{split}: {len(bad)} missing/corrupt images")\n        df["resolved_path"], df["image_sha256"] = paths, hashes\n        frames[split] = df\n        report["splits"][split] = {"count": len(df), "classes": df["2_way_label"].value_counts().to_dict(),\n                                  "missing": missing, "duplicate_ids": duplicates, "bad_images": bad}\n    names = list(frames)\n    report["overlaps"] = {}\n    for i, a in enumerate(names):\n        for b in names[i + 1:]:\n            overlap = sorted(set(frames[a].id) & set(frames[b].id))\n            report["overlaps"][f"{a}/{b}"] = overlap\n            if overlap:\n                errors.append(f"{a}/{b}: overlapping sample IDs")\n    report["errors"] = errors\n    Path(report_path).parent.mkdir(parents=True, exist_ok=True)\n    Path(report_path).write_text(json.dumps(report, indent=2), encoding="utf-8")\n    if errors:\n        raise ValueError(f"Validation failed; see {report_path}: {errors}")\n    return frames, report\n\n\nif __name__ == "__main__":\n    _, report = validate("data", "results/dataset_validation.json")\n    print(json.dumps(report, indent=2))\n', 'models.py': '"""Shared Colab/Windows model contract. Backbones are always frozen."""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom PIL import ImageOps\nfrom torchvision.models import resnet50, ResNet50_Weights\nfrom transformers import AutoModel, AutoTokenizer\n\nENCODER_CONFIG = {"text_model": "distilbert-base-uncased", "max_length": 128,\n                  "pooling": "attention_mask_mean", "image_weights": "IMAGENET1K_V2",\n                  "text_dim": 768, "image_dim": 2048, "version": 1}\nHEAD_CONFIG = {"text": {"dimensions": [768, 256, 2], "dropout": 0.3},\n               "image": {"dimensions": [2048, 256, 2], "dropout": 0.3},\n               "fusion": {"text_projection": [768, 256], "image_projection": [2048, 256],\n                          "dimensions": [512, 128, 2], "dropout": 0.3}, "version": 1}\n\n\nclass Head(nn.Module):\n    def __init__(self, kind):\n        super().__init__()\n        if kind not in ("text", "image", "fusion"):\n            raise ValueError(kind)\n        self.kind = kind\n        if kind == "fusion":\n            self.text = nn.Linear(768, 256)\n            self.image = nn.Linear(2048, 256)\n            self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 2))\n        else:\n            self.classifier = nn.Sequential(nn.Linear(768 if kind == "text" else 2048, 256),\n                                            nn.ReLU(), nn.Dropout(.3), nn.Linear(256, 2))\n\n    def forward(self, text, image):\n        x = torch.cat((self.text(text), self.image(image)), dim=1) if self.kind == "fusion" else (text if self.kind == "text" else image)\n        return self.classifier(x)\n\n\ndef masked_mean(hidden, mask):\n    mask = mask.unsqueeze(-1).to(hidden.dtype)\n    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)\n\n\nclass Encoders:\n    def __init__(self, assets, device="cpu", download=False):\n        self.device = torch.device(device)\n        assets = Path(assets)\n        text_dir = assets / "distilbert"\n        source = ENCODER_CONFIG["text_model"] if download else str(text_dir)\n        self.tokenizer = AutoTokenizer.from_pretrained(source, local_files_only=not download)\n        self.text = AutoModel.from_pretrained(source, local_files_only=not download)\n        weights = ResNet50_Weights.IMAGENET1K_V2\n        self.image = resnet50(weights=weights if download else None)\n        if download:\n            assets.mkdir(parents=True, exist_ok=True)\n            self.tokenizer.save_pretrained(text_dir)\n            self.text.save_pretrained(text_dir)\n            torch.save(self.image.state_dict(), assets / "resnet50.pt")\n        else:\n            self.image.load_state_dict(torch.load(assets / "resnet50.pt", map_location="cpu", weights_only=True))\n        self.image.fc = nn.Identity()\n        self.transform = weights.transforms()\n        for model in (self.text, self.image):\n            model.requires_grad_(False).eval().to(self.device)\n\n    @torch.inference_mode()\n    def encode_text(self, titles):\n        tokens = self.tokenizer(titles, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)\n        return masked_mean(self.text(**tokens).last_hidden_state, tokens["attention_mask"]).float()\n\n    @torch.inference_mode()\n    def encode_images(self, images):\n        batch = torch.stack([self.transform(ImageOps.exif_transpose(im).convert("RGB")) for im in images])\n        return self.image(batch.to(self.device)).float()\n', 'pipeline.py': '"""GPU-only bulk extraction/training; persistent, fingerprinted recovery files."""\nimport hashlib\nimport json\nimport os\nimport pickle\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix\nfrom models import ENCODER_CONFIG, Head\n\n\ndef atomic_save(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp)\n    os.replace(tmp, path)\n\n\ndef write_json(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".json.tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef fingerprint(frames, config):\n    records = {s: df[["id", "clean_title", "2_way_label", "image_sha256"]].to_dict("records") for s, df in frames.items()}\n    return hashlib.sha256(json.dumps({"data": records, "config": config}, sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef gpu():\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select Colab Runtime > Change runtime type > GPU, then rerun. Bulk CPU work is disabled.")\n    return torch.device("cuda")\n\n\ndef extract(frames, encoders, cache, batch_size=32):\n    gpu()\n    cache = Path(cache)\n    result = {}\n    for split, df in frames.items():\n        # Separate modality chunks let a completed text batch survive an image failure.\n        features = {}\n        for modality, dim in (("text", 768), ("image", 2048)):\n            chunks = []\n            for start in range(0, len(df), batch_size):\n                batch = df.iloc[start:start + batch_size]\n                ids = batch.id.tolist()\n                labels = torch.tensor(batch["2_way_label"].tolist(), dtype=torch.long)\n                path = cache / split / modality / f"{start:06d}-{len(batch):04d}.pt"\n                part = None\n                if path.exists():\n                    try:\n                        saved = torch.load(path, map_location="cpu", weights_only=True)\n                        if (saved["ids"] == ids and torch.equal(saved["labels"], labels)\n                                and saved["features"].shape == (len(batch), dim)\n                                and torch.isfinite(saved["features"]).all()):\n                            part = saved\n                    except (OSError, RuntimeError, EOFError, ValueError, KeyError, pickle.UnpicklingError):\n                        print("Recomputing unreadable chunk:", path)\n                if part is None:\n                    if modality == "text":\n                        embedding = encoders.encode_text(batch.clean_title.tolist())\n                    else:\n                        images = []\n                        for filename in batch.resolved_path:\n                            with Image.open(filename) as im:\n                                images.append(im.convert("RGB"))\n                        embedding = encoders.encode_images(images)\n                    part = {"ids": ids, "labels": labels, "features": embedding.cpu()}\n                    if not torch.isfinite(part["features"]).all():\n                        raise ValueError("Non-finite embeddings")\n                    atomic_save(part, path)\n                chunks.append(part["features"])\n            features[modality] = torch.cat(chunks)\n            print(f"{split}/{modality}: {len(df)} cached features ready")\n        result[split] = {**features, "labels": torch.tensor(df["2_way_label"].tolist(), dtype=torch.long), "ids": df.id.tolist()}\n    return result\n\n\ndef train(kind, features, run, config, run_id):\n    device = gpu()\n    run = Path(run)\n    last_path = run / "recovery" / f"{kind}.pt"\n    best_path = run / "models" / f"{kind}.pt"\n    seed = config["seed"]\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    model = Head(kind).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n    loss_fn = nn.CrossEntropyLoss()\n    start, best, stale, history, best_state = 0, float("inf"), 0, [], None\n    if last_path.exists():\n        state = torch.load(last_path, map_location="cpu", weights_only=True)\n        if state["run_id"] != run_id:\n            raise ValueError("Recovery fingerprint mismatch")\n        model.load_state_dict(state["state_dict"])\n        optimizer.load_state_dict(state["optimizer"])\n        for opt in optimizer.state.values():\n            for key, value in opt.items():\n                if isinstance(value, torch.Tensor):\n                    opt[key] = value.to(device)\n        start, best, stale, history = state["epoch"] + 1, state["best"], state["stale"], state["history"]\n        best_state = state["best_state"]\n        torch.set_rng_state(state["rng"])\n        torch.cuda.set_rng_state_all(state["cuda_rng"])\n        print(f"{kind}: recovering after epoch {start}")\n    data = {s: {k: v.to(device) for k, v in features[s].items() if k != "ids"} for s in ("train", "validate")}\n    for epoch in range(start, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        model.train()\n        order = torch.randperm(len(data["train"]["labels"]), device=device)\n        total = 0.0\n        for idx in order.split(config["batch_size"]):\n            d = data["train"]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(d["text"][idx], d["image"][idx]), d["labels"][idx])\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(idx)\n        model.eval()\n        with torch.inference_mode():\n            d = data["validate"]\n            val = loss_fn(model(d["text"], d["image"]), d["labels"]).item()\n        if not np.isfinite(val):\n            raise ValueError("Non-finite validation loss")\n        history.append({"epoch": epoch + 1, "train_loss": total / len(order), "val_loss": val})\n        if val < best:\n            best, stale = val, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        # One recovery file contains both last and best states, avoiding split-file recovery races.\n        atomic_save({"run_id": run_id, "epoch": epoch, "state_dict": model.state_dict(),\n                     "optimizer": optimizer.state_dict(), "best_state": best_state, "best": best,\n                     "stale": stale, "history": history, "rng": torch.get_rng_state(),\n                     "cuda_rng": torch.cuda.get_rng_state_all()}, last_path)\n        print(f"{kind} epoch {epoch + 1}: train={history[-1][\'train_loss\']:.4f} val={val:.4f}")\n    if best_state is None:\n        raise ValueError("No trained checkpoint available; epochs must be positive")\n    atomic_save({"state_dict": best_state, "kind": kind, "run_id": run_id}, best_path)\n    write_json(history, run / "results" / f"{kind}_history.json")\n    return history\n\n\ndef evaluate(features, run, run_id):\n    device = gpu()\n    run = Path(run)\n    test = features["test"]\n    assert len(test["ids"]) == 500, "Use the unchanged 500-example test split"\n    y = test["labels"].numpy()\n    metrics, predictions = {}, {"id": test["ids"], "label": y.tolist()}\n    for kind in ("text", "image", "fusion"):\n        checkpoint = torch.load(run / "models" / f"{kind}.pt", map_location="cpu", weights_only=True)\n        assert checkpoint["run_id"] == run_id\n        model = Head(kind).to(device).eval()\n        model.load_state_dict(checkpoint["state_dict"])\n        with torch.inference_mode():\n            probabilities = model(test["text"].to(device), test["image"].to(device)).softmax(1).cpu().numpy()\n        pred = probabilities.argmax(1)\n        p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=0, zero_division=0)\n        metrics[kind] = {"accuracy": float(accuracy_score(y, pred)), "precision": float(p), "recall": float(r),\n                         "f1": float(f), "roc_auc": float(roc_auc_score(y == 0, probabilities[:, 0])),\n                         "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n        predictions[f"{kind}_prediction"] = pred.tolist()\n        predictions[f"{kind}_fake_probability"] = probabilities[:, 0].tolist()\n    result = {"run_id": run_id, "test_count": 500, "positive_class": "Fake (0)", "models": metrics,\n              "fusion_outperforms_both": {metric: all(metrics["fusion"][metric] > metrics[k][metric] for k in ("text", "image"))\n                                          for metric in ("accuracy", "f1")}}\n    write_json(result, run / "results" / "metrics.json")\n    import pandas as pd\n    pd.DataFrame(predictions).to_csv(run / "results" / "predictions.csv", index=False)\n    import matplotlib\n    matplotlib.use("Agg")  # Export figures without a desktop/Tk dependency.\n    import matplotlib.pyplot as plt\n    pd.DataFrame(metrics).T[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot.bar(ylim=(0, 1), figsize=(10, 5))\n    plt.tight_layout()\n    plt.savefig(run / "results" / "comparison.png")\n    plt.close()\n    for kind in metrics:\n        fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n        matrix = np.array(metrics[kind]["confusion_matrix"])\n        axes[0].imshow(matrix, cmap="Blues")\n        for (i, j), value in np.ndenumerate(matrix):\n            axes[0].text(j, i, str(value), ha="center")\n        axes[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"], xlabel="Predicted", ylabel="Actual")\n        history = pd.read_json(run / "results" / f"{kind}_history.json")\n        axes[1].plot(history.epoch, history.train_loss, label="Train")\n        axes[1].plot(history.epoch, history.val_loss, label="Validation")\n        axes[1].legend()\n        axes[1].set(xlabel="Epoch", ylabel="Cross-entropy")\n        fig.suptitle(kind)\n        fig.tight_layout()\n        fig.savefig(run / "results" / f"{kind}.png")\n        plt.close(fig)\n    return result\n', 'inference.py': '"""Offline CPU inference from a Colab-exported bundle."""\nimport json\nfrom pathlib import Path\nimport torch\nfrom models import Encoders, Head, ENCODER_CONFIG, HEAD_CONFIG\n\n\nclass Predictor:\n    def __init__(self, bundle="artifacts"):\n        bundle = Path(bundle)\n        config = json.loads((bundle / "config.json").read_text())\n        if (config["encoders"] != ENCODER_CONFIG or config["heads"] != HEAD_CONFIG\n                or config["labels"] != {"0": "Fake", "1": "Real"}):\n            raise ValueError("Incompatible model configuration")\n        self.encoders = Encoders(bundle / "encoders", "cpu")\n        self.heads = {}\n        for kind in ("text", "image", "fusion"):\n            checkpoint = torch.load(bundle / "models" / f"{kind}.pt", map_location="cpu", weights_only=True)\n            if checkpoint["kind"] != kind or checkpoint["run_id"] != config["run_id"]:\n                raise ValueError("Mixed checkpoint bundle")\n            model = Head(kind)\n            model.load_state_dict(checkpoint["state_dict"])\n            self.heads[kind] = model.eval()\n\n    @torch.inference_mode()\n    def predict(self, title, image):\n        if not title.strip():\n            raise ValueError("Enter a headline")\n        text = self.encoders.encode_text([title.strip()])\n        visual = self.encoders.encode_images([image])\n        return {kind: model(text, visual).softmax(1)[0].tolist() for kind, model in self.heads.items()}\n', 'app.py': 'from pathlib import Path\nimport json\nimport pandas as pd\nimport streamlit as st\nfrom PIL import Image\n\nst.set_page_config(page_title="Multimodal News Analysis", page_icon="📰", layout="wide")\nBUNDLE = Path(__file__).resolve().parent / "artifacts"\nst.title("Multimodal News Analysis")\nst.caption("Academic Fakeddit prototype • Predictions reflect learned dataset patterns, not independent fact verification.")\npage = st.sidebar.radio("Explore", ["Fake News Detection", "Model Comparison", "Dataset Analysis", "Model Architecture"])\n\n\n@st.cache_resource\ndef load_predictor(bundle):\n    from inference import Predictor\n    return Predictor(bundle)\n\n\nif page == "Fake News Detection":\n    title = st.text_area("News headline or social media post")\n    uploaded = st.file_uploader("Accompanying image", type=["jpg", "jpeg", "png", "webp"])\n    if st.button("Analyze", type="primary"):\n        if not title.strip() or uploaded is None:\n            st.warning("Enter text and upload an image to compare all three models.")\n        elif not (BUNDLE / "config.json").exists():\n            st.error("Run colab_training.ipynb, download windows_demo.zip, and extract its artifacts folder here first.")\n        else:\n            try:\n                with st.spinner("Running the three models on CPU…"):\n                    image = Image.open(uploaded).convert("RGB")\n                    predictions = load_predictor(str(BUNDLE)).predict(title, image)\n                for column, (kind, probabilities) in zip(st.columns(3), predictions.items()):\n                    label = ["Fake", "Real"][int(probabilities[1] > probabilities[0])]\n                    column.metric(kind.title(), label, f"{max(probabilities):.1%} model probability")\n                st.bar_chart(pd.DataFrame(predictions, index=["Fake", "Real"]).T)\n                st.image(image, width=320)\n            except Exception as exc:\n                st.error(f"Could not load/analyze the exported bundle: {exc}")\nelif page == "Model Comparison":\n    path = BUNDLE / "results" / "metrics.json"\n    if path.exists():\n        result = json.loads(path.read_text())\n        st.caption("Same 500 test pairs for all models. Precision, recall, F1 and ROC-AUC use Fake as the positive class.")\n        st.dataframe(pd.DataFrame(result["models"]).T.drop(columns="confusion_matrix"))\n        st.write("Fusion strictly outperforms both unimodal models:", result["fusion_outperforms_both"])\n        for kind in result["models"]:\n            st.image(str(BUNDLE / "results" / f"{kind}.png"))\n        st.dataframe(pd.read_csv(BUNDLE / "results" / "predictions.csv").head(20))\n    else:\n        st.info("Evaluation results appear after Colab training and export.")\nelif page == "Dataset Analysis":\n    report_path = BUNDLE / "results" / "dataset_validation.json"\n    if not report_path.exists():\n        report_path = Path(__file__).resolve().parent / "results" / "dataset_validation.json"\n    if report_path.exists():\n        report = json.loads(report_path.read_text())\n        st.dataframe(pd.DataFrame({s: {"samples": d["count"], "Fake": d["classes"].get("0", 0), "Real": d["classes"].get("1", 0)} for s, d in report["splits"].items()}).T)\n        st.write("Existing clean_title and paired local images are used without resampling or relabeling.")\n        st.json(report)\n    data_root = Path(__file__).resolve().parent / "data"\n    if (data_root / "test_processed.csv").exists():\n        from dataset import resolve_image\n        samples = pd.read_csv(data_root / "test_processed.csv", dtype={"id": str})\n        selected = st.selectbox("Test sample", range(len(samples)))\n        row = samples.iloc[selected]\n        st.write(row.clean_title)\n        st.write("Dataset label:", {0: "Fake", 1: "Real"}[int(row["2_way_label"])])\n        st.image(str(resolve_image(data_root, row.local_image_path)), width=300)\nelse:\n    st.graphviz_chart(\'digraph { rankdir=LR; Text -> DistilBERT -> "768 features"; Image -> ResNet50 -> "2048 features"; "768 features" -> "Text head"; "2048 features" -> "Image head"; "768 features" -> "256 text projection" -> Concatenate; "2048 features" -> "256 image projection" -> Concatenate; Concatenate -> "512 → 128 → 2 fusion head"; }\')\n    st.write("DistilBERT uses attention-mask-aware mean pooling (128-token limit). ResNet50 uses IMAGENET1K_V2 transforms and global pooled features. Both pretrained encoders stay frozen.")\n    st.write("Unimodal heads: input → 256 → ReLU → Dropout(0.3) → 2. Fusion: separate 256-dimensional projections → concatenation → 128 → ReLU → Dropout(0.3) → 2.")\n    st.write("Feature extraction and head training run on a Colab GPU. This app loads the exported encoders and heads on CPU without network downloads.")\n', 'requirements.txt': '# Install CPU torch/torchvision separately on Windows; see README.\ntorch>=2.6,<3\ntorchvision>=0.21,<1\ntransformers==4.57.1\npandas>=2.2,<3\nnumpy>=1.26,<3\nPillow>=10,<13\nscikit-learn>=1.5,<2\nmatplotlib>=3.8,<4\nstreamlit>=1.40,<2\nsafetensors>=0.4\n'}
for name, source in SOURCE_FILES.items():
    (WORK / name).write_text(source, encoding="utf-8")


## 3. Configure data source and persistent output
`drive`: point DATA_DIR at the folder containing train_processed.csv and images/.
`zip`: upload fakeddit_processed.zip when prompted, or set ZIP_ON_DRIVE to an existing ZIP path.
Use package_data.py on Windows to create the ZIP without downloading anything.
Only settings in this cell need editing. Existing CSVs are never rewritten.


In [ ]:
DATA_SOURCE = "drive"  # "drive" or "zip"
DATA_DIR = Path('/content/drive/MyDrive/Multimodal_Fake_News_Detection/data')
ZIP_ON_DRIVE = ""  # e.g. /content/drive/MyDrive/fakeddit_processed.zip
OUTPUT = Path('/content/drive/MyDrive/Fakeddit_Demo')
TRAIN_CONFIG = dict(seed=42, epochs=25, batch_size=64, lr=0.001, weight_decay=0.0001, patience=5)
FEATURE_BATCH_SIZE = 32  # Lower to 16 or 8 if GPU runs out of memory.
OUTPUT.mkdir(parents=True, exist_ok=True)
from dataset import extract_zip, validate
if DATA_SOURCE == "zip":
    if ZIP_ON_DRIVE:
        archive = Path(ZIP_ON_DRIVE)
    else:
        uploaded = files.upload()
        assert len(uploaded) == 1, "Upload exactly one processed dataset ZIP."
        archive = WORK / next(iter(uploaded))
        del uploaded
    DATA_DIR = extract_zip(archive, Path('/content/fakeddit_data'))
elif DATA_SOURCE != "drive":
    raise ValueError('DATA_SOURCE must be drive or zip')
assert (DATA_DIR / 'train_processed.csv').is_file(), f'Check DATA_DIR: {DATA_DIR}'


## 4. Validate every pair and preserve the existing splits
Checks counts, labels, missing fields, duplicate IDs, overlap, image existence and decoding.
Stops on errors without dropping rows. Windows backslashes and absolute paths relocate via images/.
Binary labels stay 0=Fake, 1=Real; verify against the supplied 6_way_label true category (0).
References: [official Fakeddit repository](https://github.com/entitize/Fakeddit) and [paper](https://aclanthology.org/2020.lrec-1.755/).


In [ ]:
from pipeline import fingerprint, write_json
from models import ENCODER_CONFIG, HEAD_CONFIG
frames, report = validate(DATA_DIR, OUTPUT / 'dataset_validation.json')
FEATURE_ID = fingerprint(frames, ENCODER_CONFIG)
RUN_ID = fingerprint(frames, {"encoders": ENCODER_CONFIG, "heads": HEAD_CONFIG, "training": TRAIN_CONFIG})
RUN = OUTPUT / 'runs' / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
write_json(report, RUN / 'results' / 'dataset_validation.json')
config = {"run_id": RUN_ID, "feature_id": FEATURE_ID, "encoders": ENCODER_CONFIG, "heads": HEAD_CONFIG,
          "training": TRAIN_CONFIG, "labels": {"0": "Fake", "1": "Real"}}
write_json(config, RUN / 'config.json')
print(json.dumps(report, indent=2))
print('Persistent run:', RUN)


## 5. Load frozen pretrained encoders and extract/reuse GPU features
Each completed modality batch is persisted to Drive. Restart cells 1–5 to resume. Changing training settings reuses the same feature cache. Keep FEATURE_BATCH_SIZE unchanged when recovering feature extraction.

In [ ]:
from models import Encoders
from pipeline import extract
ASSETS = OUTPUT / 'encoders_v1'
ready = ASSETS / 'READY.json'
encoders = Encoders(ASSETS, device='cuda', download=not ready.exists())
if ready.exists():
    assert json.loads(ready.read_text()) == ENCODER_CONFIG
else:
    write_json(ENCODER_CONFIG, ready)
assert all(not p.requires_grad for m in (encoders.text, encoders.image) for p in m.parameters())
features = extract(frames, encoders, OUTPUT / 'features' / FEATURE_ID, FEATURE_BATCH_SIZE)
del encoders
import gc
gc.collect()
torch.cuda.empty_cache()


## 6. Smoke-test shapes and serialization before training

In [ ]:
from models import Head, masked_mean
import tempfile
assert torch.equal(masked_mean(torch.tensor([[[1., 2.], [99., 99.]]]), torch.tensor([[1, 0]])), torch.tensor([[1., 2.]]))
for split, data in features.items():
    assert data['text'].shape == (len(frames[split]), 768)
    assert data['image'].shape == (len(frames[split]), 2048)
    assert data['ids'] == frames[split].id.tolist()
for kind in ('text', 'image', 'fusion'):
    head = Head(kind).eval()
    t, i = features['train']['text'][:2], features['train']['image'][:2]
    assert head(t, i).shape == (2, 2)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'head.pt'
        torch.save(head.state_dict(), path)
        restored = Head(kind).eval()
        restored.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
        torch.testing.assert_close(head(t, i), restored(t, i))
print('Shape, masked pooling and CPU checkpoint round-trip checks passed.')


## 7. Train three independent heads on GPU
Only train fits parameters; validate selects the lowest-loss checkpoint. Recovery includes optimizer, RNG, early stopping and the best weights. An interrupted epoch is replayed; completed epochs are retained. Completed runs are skipped automatically.

In [ ]:
from pipeline import train
for kind in ('text', 'image', 'fusion'):
    train(kind, features, RUN, TRAIN_CONFIG, RUN_ID)


## 8. Evaluate all models on the same unchanged 500 test examples
Fake (0) is the positive class for precision/recall/F1/AUC. No test-based model selection. Results below are measured, never placeholders.

In [ ]:
from pipeline import evaluate
metrics = evaluate(features, RUN, RUN_ID)
print(json.dumps(metrics, indent=2))
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN / 'results' / 'comparison.png')))


## 9. Export encoders, heads, configs and results; verify offline CPU inference
The Windows bundle contains both pretrained backbones so the demo needs no model downloads. A single-example CPU check compares exported inference with the GPU cached-feature prediction.

In [ ]:
import importlib.metadata
import zipfile
from PIL import Image
from inference import Predictor
BUNDLE = Path('/content/windows_demo/artifacts')
BUNDLE.mkdir(parents=True, exist_ok=True)
for folder in ('models', 'results'):
    shutil.copytree(RUN / folder, BUNDLE / folder, dirs_exist_ok=True)
shutil.copy2(RUN / 'config.json', BUNDLE / 'config.json')
shutil.copytree(ASSETS, BUNDLE / 'encoders', dirs_exist_ok=True)
versions = {p: importlib.metadata.version(p) for p in ('torch', 'torchvision', 'transformers', 'numpy', 'pandas', 'Pillow', 'scikit-learn')}
write_json(versions, BUNDLE / 'environment.json')
torch.set_num_threads(2)
predictor = Predictor(BUNDLE)
row = frames['test'].iloc[0]
with Image.open(row.resolved_path) as im:
    prediction = predictor.predict(row.clean_title, im)
for kind, head in predictor.heads.items():
    with torch.inference_mode():
        expected = head(features['test']['text'][:1], features['test']['image'][:1]).softmax(1)[0]
    torch.testing.assert_close(torch.tensor(prediction[kind]), expected, atol=2e-3, rtol=2e-3)
write_json({'passed': True, 'sample_id': row.id, 'probabilities': prediction}, BUNDLE / 'results' / 'cpu_smoke.json')
for name in ('app.py', 'models.py', 'dataset.py', 'inference.py', 'requirements.txt'):
    shutil.copy2(WORK / name, BUNDLE.parent / name)
export_path = OUTPUT / 'windows_demo.zip'
temporary = export_path.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in BUNDLE.parent.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(BUNDLE.parent).as_posix())
os.replace(temporary, export_path)
shutil.copy2(BUNDLE / 'results' / 'cpu_smoke.json', RUN / 'results' / 'cpu_smoke.json')
print('CPU export check passed. Download from Google Drive:', export_path)
print('Extract ZIP into your Windows project; then run: python -m streamlit run app.py')


## 10. Optional browser download
The ZIP is already saved in Drive. For large bundles, downloading through the Drive website is more reliable.
After a Colab disconnect rerun the cells in order with identical settings. Never delete Drive recovery/cache folders to resume.
No full training or feature extraction is required on Windows.


In [ ]:
# Optional:
# files.download(str(export_path))
